# 🤖 Notebook 03 — Predictive Modeling (Random Forest)
**CT Guanabara Futevolei | Data Science Performance Project**

This notebook trains and evaluates a **Random Forest Regressor** to predict
week-over-week IPG improvement based on training variables.

**Target:** `delta_ipg_pct` — percentage change in IPG from one week to the next  
**Features:** `sessions_per_week`, `rest_days`, `bmi`, `ipg_prev_week`, `experience_years`


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, r2_score
from feature_engineering import build_model_dataset, FEATURE_COLS, TARGET_COL

X, y, model_df = build_model_dataset()
print(f"✅ Model dataset ready: {X.shape[0]} samples × {X.shape[1]} features")
print(f"   Target: {TARGET_COL}")
print(f"   Features: {FEATURE_COLS}")


## 1. Feature-Target Correlations

In [ ]:
corr = X.corrwith(y).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#1B4332' if v > 0 else '#E76F51' for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='#6C757D', lw=1)
ax.set_xlabel('Pearson Correlation with Δ IPG (%)', fontsize=11)
ax.set_title('Feature Correlations with Target Variable',
             fontsize=12, fontweight='bold', color='#1B4332')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr.round(3))


## 2. Model Training — 5-Fold Cross-Validation

In [ ]:
RF_PARAMS = {
    'n_estimators': 100,
    'max_depth': 5,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42,
}

model = RandomForestRegressor(**RF_PARAMS)
kf    = KFold(n_splits=5, shuffle=True, random_state=42)

cv = cross_validate(model, X, y, cv=kf,
                    scoring=['neg_root_mean_squared_error','r2'],
                    return_train_score=True)

rmse_tr = -cv['train_neg_root_mean_squared_error']
rmse_v  = -cv['test_neg_root_mean_squared_error']
r2_tr   =  cv['train_r2']
r2_v    =  cv['test_r2']

results = pd.DataFrame({
    'Fold':       [f'Fold {i+1}' for i in range(5)],
    'RMSE Train': rmse_tr.round(4),
    'RMSE Val':   rmse_v.round(4),
    'R² Train':   r2_tr.round(4),
    'R² Val':     r2_v.round(4),
})
results.loc['Mean'] = ['Mean', rmse_tr.mean().round(4), rmse_v.mean().round(4),
                        r2_tr.mean().round(4), r2_v.mean().round(4)]
results.loc['Std']  = ['Std',  rmse_tr.std().round(4),  rmse_v.std().round(4),
                        r2_tr.std().round(4),  r2_v.std().round(4)]
display(results.set_index('Fold'))


## 3. Cross-Validation Results — Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(5); w = 0.35
folds = [f'Fold {i+1}' for i in range(5)]

axes[0].bar(x-w/2, rmse_tr, w, label='Train', color='#2D6A4F')
axes[0].bar(x+w/2, rmse_v,  w, label='Validation', color='#E76F51')
axes[0].axhline(rmse_v.mean(), color='#E76F51', linestyle='--', alpha=0.6,
                label=f'Val mean: {rmse_v.mean():.3f}')
axes[0].set_xticks(x); axes[0].set_xticklabels(folds)
axes[0].set_title('RMSE per Fold', fontweight='bold')
axes[0].set_ylabel('RMSE'); axes[0].legend(frameon=False)
axes[0].spines[['top','right']].set_visible(False)

axes[1].bar(x-w/2, r2_tr, w, label='Train', color='#2D6A4F')
axes[1].bar(x+w/2, r2_v,  w, label='Validation', color='#E76F51')
axes[1].axhline(r2_v.mean(), color='#E76F51', linestyle='--', alpha=0.6,
                label=f'Val mean: {r2_v.mean():.3f}')
axes[1].set_xticks(x); axes[1].set_xticklabels(folds)
axes[1].set_title('R² per Fold', fontweight='bold')
axes[1].set_ylabel('R²'); axes[1].legend(frameon=False); axes[1].set_ylim(0, 1.05)
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Random Forest — 5-Fold Cross-Validation Results',
             fontsize=14, fontweight='bold', color='#1B4332')
plt.tight_layout()
plt.savefig('../outputs/figures/cv_results.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Feature Importance

In [ ]:
model.fit(X, y)
fi = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
palette = ['#1B4332','#2D6A4F','#52B788','#74C69D','#95D5B2']
bars = ax.barh(fi.index[::-1], fi.values[::-1],
               color=palette[::-1][:len(fi)], edgecolor='white')
for bar, val in zip(bars, fi.values[::-1]):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=10, color='#1B4332')
ax.set_xlabel('Relative Importance'); ax.set_xlim(0, fi.max()*1.25)
ax.set_title('Feature Importance — Random Forest Regressor',
             fontsize=12, fontweight='bold', color='#1B4332')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFeature Importance:")
for f, v in fi.items():
    print(f"  {f:<30} {v*100:.1f}%")


## 5. Actual vs Predicted

In [ ]:
y_pred = model.predict(X)
rmse_full = np.sqrt(mean_squared_error(y, y_pred))
r2_full   = r2_score(y, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y, y_pred, color='#2D6A4F', alpha=0.75, edgecolors='white', s=80)
lim = [min(y.min(), y_pred.min())-0.3, max(y.max(), y_pred.max())+0.3]
ax.plot(lim, lim, '--', color='#E76F51', lw=1.5, label='Ideal prediction')
ax.set_xlabel('Actual Δ IPG (%)', fontsize=12)
ax.set_ylabel('Predicted Δ IPG (%)', fontsize=12)
ax.set_title(f'Actual vs Predicted | RMSE: {rmse_full:.3f} | R²: {r2_full:.3f}',
             fontsize=12, fontweight='bold', color='#1B4332')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
